# Dataset explorer

Set `dataset_dir` to an existing Parquet dataset directory. Loading returns a Polars LazyFrame; only the preview cell reads rows. Run once in this environment: `pip install -e ".[notebooks,vis]"`. The `vis` extra includes Kaleido for the SVG renderer and static image export.

In [4]:
import matplotlib.pyplot as plt
import pandas as pd
import polars as pl
import plotly.io as pio

from pathlib import Path

from comm_research.infra.paths import PROJECT_ROOT
from comm_research.infra.lake.parquet import scan_parquet
from comm_research.infra.tools.vis import *

In [5]:
DATA_DIR = PROJECT_ROOT / "data/lake/weather/era5/t_2m"  

In [ ]:
def load_data(data_dir):

    path = Path(DATA_DIR)
    return scan_parquet(path).collect()

def save(df: pl.DataFrame | pl.LazyFrame, filepath: str | Path) -> Path:
    """Write Parquet to a relative path under data/clean and return its path."""
    clean_dir = (PROJECT_ROOT / "data" / "clean").resolve()
    relative_path = Path(filepath)
    if relative_path.is_absolute():
        raise ValueError("filepath must be relative to data/clean")

    path = (clean_dir / relative_path).with_suffix(".parquet").resolve()
    if not path.is_relative_to(clean_dir):
        raise ValueError("filepath must stay within data/clean")

    path.parent.mkdir(parents=True, exist_ok=True)
    if isinstance(df, pl.LazyFrame):
        df.sink_parquet(path, compression="zstd")
    else:
        df.write_parquet(path, compression="zstd")
    return path


In [10]:
def socal_temp_2m(dataset_dir, lon, lat):
    table = scan_parquet(Path(dataset_dir))
    df = table.collect()

    df = df.filter(
        (pl.col('latitude') == lat) &
        (pl.col('longitude') == lon)
    )
    return df

In [ ]:
lat = 32.5
lon = -121.0

save_path = f'weather/socal/temp_2m_{lat:g}_{lon:g}.parquet'

df = socal_temp_2m(
    DATA_DIR, 
    lon, 
    lat
)

save(df, save_path)